# Día 16 — Reinforcement Learning, Sim2Real y agentes que aprenden actuando

En este cuaderno vamos a ver **ideas básicas de aprendizaje por refuerzo** usando entornos pequeños de `Gymnasium`.

La meta no es convertirnos en expertos en RL en una sesión. La meta es entender el cambio de paradigma:

> En aprendizaje por refuerzo, el modelo no solo predice: **actúa**, observa consecuencias y mejora una **política**.

Trabajaremos con cuatro bloques: un ejemplo y un ejercicio de control continuo, y un ejemplo y un ejercicio con espacios discretos. En cada bloque compararemos:

1. **Política aleatoria:** línea base sin inteligencia.
2. **Política heurística:** regla diseñada a mano con intuición de ingeniería.
3. **Política aprendida:** parámetros ajustados por recompensa.

## Objetivos de aprendizaje

Al terminar el cuaderno deberías poder:

- identificar agente, entorno, estado, acción, recompensa y episodio;
- explicar por qué una recompensa cambia el comportamiento aprendido;
- comparar una política aleatoria, una heurística y una política aprendida;
- distinguir cuándo el estado/acción son continuos, discretos o discretizados;
- interpretar momentos intermedios del entrenamiento, no solo el resultado final;
- reconocer riesgos de transferir una política de simulación a un sistema físico.


## 0. Instalación recomendada

Este cuaderno está pensado para ejecutarse desde el repositorio `RLTutorials`, usando el entorno virtual creado con `requirements.txt` y `requirements-box2d.txt`.

Si estás siguiendo el README, no necesitas instalar nada desde el cuaderno. La celda siguiente queda como recordatorio para entornos temporales como Colab o una máquina recién configurada.


In [ ]:
# Ejecuta esta celda solo si necesitas instalar dependencias.
# En Colab suele funcionar. En Windows, Box2D puede requerir instalación adicional de swig.

# %pip install -q "gymnasium[classic-control]" "gymnasium[mujoco]" "gymnasium[box2d]" ipywidgets matplotlib pandas numpy


## 1. Importar librerías

Como el repositorio ya declara sus dependencias, asumimos que `gymnasium`, `mujoco`, `box2d`, `ipywidgets`, `numpy`, `pandas` y `matplotlib` están instalados.

Esta celda solo carga las librerías que usaremos en el resto del cuaderno.


In [ ]:
import math
import time
from dataclasses import dataclass
from typing import Callable, Tuple

import gymnasium as gym
import ipywidgets as widgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML, Markdown

# Estilo visual común para todas las gráficas del cuaderno.
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

print("Dependencias principales importadas correctamente.")


## 2. Funciones auxiliares

En lugar de esconder todo en un bloque grande, iremos función por función. La idea es que puedas ubicar rápidamente qué pieza hace qué.


### Función auxiliar: `run_episode`

**Qué hace:** ejecuta un episodio completo de Gymnasium.

**Parámetros principales:**

- `env_id`: nombre del entorno, por ejemplo `"MountainCar-v0"`.
- `policy_fn`: función que recibe una observación y devuelve una acción.
- `seed`: semilla para reproducibilidad.
- `max_steps`: límite de pasos del episodio.
- `render`: si es `True`, guarda frames para construir video.

**Cómo se llama:**

```python
total, steps, frames = run_episode("MountainCar-v0", policy, seed=0, max_steps=200, render=True)
```


In [ ]:
def run_episode(env_id: str,
                policy_fn: Callable[[np.ndarray], object],
                seed: int = 0,
                max_steps: int = 500,
                render: bool = False):
    '''Ejecuta un episodio completo y devuelve recompensa, pasos y frames.'''
    render_mode = "rgb_array" if render else None
    env = gym.make(env_id, render_mode=render_mode)

    obs, info = env.reset(seed=seed)
    total_reward = 0.0
    frames = []

    for t in range(max_steps):
        if render:
            frames.append(env.render())

        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)

        if terminated or truncated:
            break

    env.close()
    return total_reward, t + 1, frames


### Función auxiliar: `random_policy_factory`

**Qué hace:** crea una política aleatoria compatible con el espacio de acciones del entorno.

**Parámetros:**

- `env_id`: entorno del que se leerá el espacio de acciones.

**Cómo se llama:**

```python
random_policy = random_policy_factory("LunarLander-v3")
```


In [ ]:
def random_policy_factory(env_id: str):
    '''Crea una política aleatoria compatible con el espacio de acciones del entorno.'''
    env = gym.make(env_id)
    action_space = env.action_space
    env.close()

    def policy(obs):
        # Ignora la observación: es nuestra línea base mínima.
        return action_space.sample()

    return policy


### Función auxiliar: `evaluate_policy`

**Qué hace:** corre una política varias veces y devuelve una tabla con recompensa y duración de cada episodio.

**Parámetros:**

- `env_id`: entorno.
- `policy_fn`: política a evaluar.
- `episodes`: número de episodios.
- `max_steps`: pasos máximos por episodio.
- `seed`: semilla base.

**Cómo se llama:**

```python
df = evaluate_policy("InvertedPendulum-v5", policy, episodes=5, max_steps=300)
```


In [ ]:
def evaluate_policy(env_id: str,
                    policy_fn: Callable[[np.ndarray], object],
                    episodes: int = 10,
                    max_steps: int = 500,
                    seed: int = 0) -> pd.DataFrame:
    '''Evalúa una política varias veces y devuelve una tabla de resultados.'''
    results = []
    for ep in range(episodes):
        total, steps, _ = run_episode(env_id, policy_fn, seed=seed + ep, max_steps=max_steps)
        results.append({"episodio": ep, "recompensa": total, "pasos": steps})
    return pd.DataFrame(results)


### Función auxiliar: `score_policy`

**Qué hace:** resume una política en un solo número: recompensa promedio.

**Parámetros:** los mismos de `evaluate_policy`, pero devuelve un `float`.

**Cómo se llama:**

```python
score = score_policy("MountainCar-v0", policy, episodes=3)
```


In [ ]:
def score_policy(env_id: str,
                 policy_fn: Callable[[np.ndarray], object],
                 episodes: int = 3,
                 max_steps: int = 500,
                 seed: int = 0) -> float:
    '''Resume una política en un solo número: recompensa promedio.'''
    df = evaluate_policy(env_id, policy_fn, episodes=episodes, max_steps=max_steps, seed=seed)
    return float(df["recompensa"].mean())


### Función auxiliar: `plot_rewards`

**Qué hace:** grafica una columna de recompensas por episodio.

**Parámetros:**

- `df`: tabla con columnas `episodio` y `recompensa`.
- `title`: título de la gráfica.

**Cómo se llama:**

```python
plot_rewards(df, "Recompensa por episodio")
```


In [ ]:
def plot_rewards(df: pd.DataFrame, title: str = "Recompensa por episodio"):
    '''Grafica recompensas episodio a episodio.'''
    fig, ax = plt.subplots()
    ax.plot(df["episodio"], df["recompensa"], marker="o", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    plt.show()


### Función auxiliar: `moving_average`

**Qué hace:** suaviza una curva ruidosa usando promedio móvil.

**Parámetros:**

- `x`: lista o arreglo numérico.
- `window`: tamaño de ventana.

**Cómo se llama:**

```python
ma = moving_average(rewards, window=50)
```


In [ ]:
def moving_average(x, window=50):
    '''Suaviza una curva para ver tendencia sin tanto ruido.'''
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")


### Función auxiliar: `show_episode_animation`

**Qué hace:** convierte los frames de un episodio en un reproductor HTML dentro del notebook.

**Parámetros principales:**

- `env_id`: entorno.
- `policy_fn`: política a visualizar.
- `max_steps`: duración máxima.
- `frame_skip`: reduce peso del HTML saltando frames.
- `pad_to_frames`: repite el último frame si el episodio termina muy pronto.
- `done_label`: texto superpuesto cuando el episodio terminó antes de tiempo, por ejemplo `"CAÍDA"`.

**Cómo se llama:**

```python
show_episode_animation("InvertedPendulum-v5", policy, done_label="CAÍDA")
```


In [ ]:
def show_episode_animation(env_id,
                           policy_fn,
                           seed=0,
                           max_steps=300,
                           fps=30,
                           title=None,
                           frame_skip=2,
                           pad_to_frames=None,
                           done_label=None):
    '''Muestra una animación corta de una política dentro del notebook.'''
    if title:
        display(Markdown(f"**{title}**"))

    total, steps, frames = run_episode(env_id, policy_fn, seed=seed, max_steps=max_steps, render=True)
    ended_early = steps < max_steps
    overlay_flags = [False] * len(frames)

    if done_label and ended_early and overlay_flags:
        overlay_flags[-1] = True

    if pad_to_frames is not None and frames:
        while len(frames) < pad_to_frames:
            frames.append(frames[-1])
            overlay_flags.append(bool(done_label and ended_early))

    if frame_skip and frame_skip > 1 and len(frames) > 2:
        frame_pairs = list(zip(frames, overlay_flags))
        reduced_pairs = frame_pairs[::frame_skip]
        if reduced_pairs[-1][0] is not frames[-1]:
            reduced_pairs.append(frame_pairs[-1])
        frames = [frame for frame, _ in reduced_pairs]
        overlay_flags = [flag for _, flag in reduced_pairs]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    im = ax.imshow(frames[0])
    done_text = ax.text(
        0.5,
        0.12,
        done_label or "",
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=20,
        fontweight="bold",
        color="white",
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "crimson", "alpha": 0.85},
        visible=False,
    )

    def update(i):
        im.set_array(frames[i])
        done_text.set_visible(bool(done_label and overlay_flags[i]))
        return [im, done_text]

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml(default_mode="once")))
    print(f"Recompensa total: {total:.2f} | pasos simulados: {steps} | frames mostrados: {len(frames)}")


### Función auxiliar: `show_training_history_viewer`

**Qué hace:** crea un selector para visualizar políticas intermedias guardadas durante el entrenamiento.

**Parámetros principales:**

- `env_id`: entorno.
- `snapshots`: lista de momentos guardados.
- `policy_builder`: función que convierte parámetros en política.
- `param_names`: nombres legibles de parámetros.

**Cómo se llama:**

```python
show_training_history_viewer(env_id, snapshots, policy_builder)
```


In [ ]:
def show_training_history_viewer(env_id,
                                 snapshots,
                                 policy_builder,
                                 seed=0,
                                 max_steps=300,
                                 fps=30,
                                 frame_skip=2,
                                 pad_to_frames=None,
                                 done_label=None,
                                 param_names=None):
    '''Crea un selector manual para ver políticas intermedias del entrenamiento.'''
    if not snapshots:
        print("No hay historial de políticas para mostrar.")
        return

    options = [
        (f"iteración {snap['iteración']} | recompensa {snap['recompensa']:.1f}", idx)
        for idx, snap in enumerate(snapshots)
    ]

    def render_snapshot(momento):
        snap = snapshots[momento]
        params = np.asarray(snap["params"], dtype=float)
        names = param_names if param_names is not None and len(param_names) == len(params) else [f"p{i}" for i in range(len(params))]

        display(pd.DataFrame({
            "parámetro": names,
            "valor": params,
        }))
        print(f"Mostrando mejor política de la generación {snap['iteración']} | recompensa: {snap['recompensa']:.2f}")

        show_episode_animation(
            env_id,
            policy_builder(params),
            seed=seed,
            max_steps=max_steps,
            fps=fps,
            title=f"Momento del entrenamiento: iteración {snap['iteración']}",
            frame_skip=frame_skip,
            pad_to_frames=pad_to_frames,
            done_label=done_label,
        )

    display(widgets.interactive(
        render_snapshot,
        {"manual": True, "manual_name": "Ver momento"},
        momento=widgets.Dropdown(options=options, value=len(options) - 1, description="momento")
    ))


### Función auxiliar: `show_qlearning_history_viewer`

**Qué hace:** permite elegir una etapa del entrenamiento de Q-learning y ver la política greedy asociada a la tabla $Q$ de ese momento.

**Parámetros principales:**

- `env_id`: entorno que se va a visualizar.
- `snapshots`: lista de tablas $Q$ guardadas durante el entrenamiento.
- `max_steps`: pasos máximos del episodio mostrado.
- `seed`: semilla para comparar políticas de forma reproducible.

**Cómo se llama:**

```python
show_qlearning_history_viewer("MountainCar-v0", mountain_q_snapshots)
```


In [ ]:
def show_qlearning_history_viewer(env_id,
                                  snapshots,
                                  max_steps=200,
                                  seed=0,
                                  fps=30,
                                  frame_skip=2):
    '''Muestra políticas intermedias guardadas durante Q-learning.'''
    if not snapshots:
        print("No hay snapshots de entrenamiento para mostrar.")
        return

    options = []
    for index, snapshot in enumerate(snapshots):
        label = f"episodio {snapshot['episodio']} | recompensa reciente {snapshot['recompensa_reciente']:.1f}"
        options.append((label, index))

    selector = widgets.Dropdown(
        options=options,
        value=len(options) - 1,
        description="momento",
        layout=widgets.Layout(width="520px"),
    )
    output = widgets.Output()

    def render_snapshot(change=None):
        with output:
            output.clear_output(wait=True)
            snapshot = snapshots[selector.value]
            policy = mountaincar_policy_from_Q(snapshot["Q"], snapshot["discretizer"])
            display(Markdown(
                f"**Episodio guardado:** {snapshot['episodio']}  \\n"
                f"**Recompensa promedio reciente:** {snapshot['recompensa_reciente']:.1f}"
            ))
            show_episode_animation(
                env_id,
                policy,
                seed=seed,
                max_steps=max_steps,
                fps=fps,
                frame_skip=frame_skip,
            )

    selector.observe(render_snapshot, names="value")
    display(selector, output)
    render_snapshot()


### Función auxiliar: `compare_policy_summary`

**Qué hace:** construye una tabla comparativa final entre políticas.

**Parámetros:**

- `env_id`: entorno.
- `policies`: lista de pares `(nombre, política)`.
- `episodes`: número de episodios para evaluar.
- `max_steps`: límite de pasos.

**Cómo se llama:**

```python
compare_policy_summary(env_id, [("aleatoria", random_policy), ("heurística", heuristic)])
```


In [ ]:
def compare_policy_summary(env_id, policies, episodes=5, max_steps=500, seed=0):
    rows = []
    for name, policy in policies:
        df = evaluate_policy(env_id, policy, episodes=episodes, max_steps=max_steps, seed=seed)
        rows.append({
            "entorno": env_id,
            "política": name,
            "recompensa_promedio": df["recompensa"].mean(),
            "pasos_promedio": df["pasos"].mean(),
        })
    return pd.DataFrame(rows)


### Función auxiliar: `evaluate_policy_objectives`

**Qué hace:** evalúa una política con dos lentes: la recompensa original del ambiente y una recompensa moldeada opcional.

**Parámetros principales:**

- `env_id`: nombre del entorno.
- `policy_fn`: política que se quiere evaluar.
- `shaped_reward_fn`: función que calcula la recompensa moldeada.
- `env_kwargs`: argumentos extra para crear entornos como `FrozenLake-v1`.

**Cómo se llama:**

```python
df = evaluate_policy_objectives(env_id, policy, shaped_reward_fn=mi_recompensa)
```


In [ ]:
def evaluate_policy_objectives(env_id,
                               policy_fn,
                               shaped_reward_fn=None,
                               episodes=5,
                               max_steps=500,
                               seed=0,
                               env_kwargs=None) -> pd.DataFrame:
    '''Evalúa recompensa original y recompensa moldeada para la misma política.'''
    env_kwargs = dict(env_kwargs or {})
    rows = []

    for ep in range(episodes):
        env = gym.make(env_id, **env_kwargs)
        obs, info = env.reset(seed=seed + ep)
        original_total = 0.0
        shaped_total = 0.0

        for t in range(max_steps):
            action = policy_fn(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)

            original_reward = float(reward)
            if shaped_reward_fn is None:
                shaped_reward = original_reward
            else:
                shaped_reward = float(shaped_reward_fn(
                    obs, action, next_obs, reward, terminated, truncated, info, t
                ))

            original_total += original_reward
            shaped_total += shaped_reward
            obs = next_obs

            if terminated or truncated:
                break

        env.close()
        rows.append({
            "episodio": ep,
            "recompensa_original": original_total,
            "recompensa_moldeada": shaped_total,
            "pasos": t + 1,
        })

    return pd.DataFrame(rows)


### Función auxiliar: `score_policy_objective`

**Qué hace:** resume una política en un solo número usando la recompensa original o la moldeada.

**Parámetros:** los mismos de `evaluate_policy_objectives`.

**Cómo se llama:**

```python
score = score_policy_objective(env_id, policy, shaped_reward_fn=mi_recompensa)
```


In [ ]:
def score_policy_objective(env_id,
                           policy_fn,
                           shaped_reward_fn=None,
                           episodes=3,
                           max_steps=500,
                           seed=0,
                           env_kwargs=None) -> float:
    '''Devuelve la recompensa promedio según el objetivo elegido.'''
    df = evaluate_policy_objectives(
        env_id,
        policy_fn,
        shaped_reward_fn=shaped_reward_fn,
        episodes=episodes,
        max_steps=max_steps,
        seed=seed,
        env_kwargs=env_kwargs,
    )
    column = "recompensa_moldeada" if shaped_reward_fn is not None else "recompensa_original"
    return float(df[column].mean())


### Función auxiliar: `show_objective_comparison`

**Qué hace:** compara una o varias políticas mostrando recompensa original, recompensa moldeada y duración promedio.

**Parámetros:**

- `policies`: lista de pares `(nombre, política)`.
- `shaped_reward_fn`: recompensa alternativa informada.

**Cómo se llama:**

```python
show_objective_comparison(env_id, [("aprendida", policy)], shaped_reward_fn)
```


In [ ]:
def show_objective_comparison(env_id,
                              policies,
                              shaped_reward_fn,
                              episodes=5,
                              max_steps=500,
                              seed=0,
                              env_kwargs=None):
    '''Construye una tabla comparando objetivo original y objetivo moldeado.'''
    rows = []
    for name, policy in policies:
        df = evaluate_policy_objectives(
            env_id,
            policy,
            shaped_reward_fn=shaped_reward_fn,
            episodes=episodes,
            max_steps=max_steps,
            seed=seed,
            env_kwargs=env_kwargs,
        )
        rows.append({
            "política": name,
            "recompensa_original_promedio": df["recompensa_original"].mean(),
            "recompensa_moldeada_promedio": df["recompensa_moldeada"].mean(),
            "pasos_promedio": df["pasos"].mean(),
        })

    table = pd.DataFrame(rows)
    display(table)
    return table


### Función auxiliar: `make_training_progress`

**Qué hace:** crea una barra de progreso sencilla para entrenamientos interactivos.

**Parámetros:**

- `total`: número total de iteraciones o episodios.
- `description`: texto corto para la barra.

**Cómo se llama:**

```python
progress = make_training_progress(100, "Q-learning")
progress.value = 50
```


In [ ]:
def make_training_progress(total, description="entrenando"):
    '''Crea una barra de progreso ligera para el notebook.'''
    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=int(total),
        description=description,
        bar_style="info",
        layout=widgets.Layout(width="520px"),
    )
    display(progress)
    return progress


### Función auxiliar: `show_frozenlake_episode_animation`

**Qué hace:** muestra una trayectoria de `FrozenLake-v1` usando el render del ambiente.

**Parámetros principales:**

- `policy_fn`: política discreta que recibe un estado y devuelve una acción.
- `env_kwargs`: mapa y configuración del lago.

**Cómo se llama:**

```python
show_frozenlake_episode_animation(policy)
```


In [ ]:
def show_frozenlake_episode_animation(policy_fn,
                                      seed=0,
                                      max_steps=50,
                                      fps=2,
                                      env_kwargs=None,
                                      title=None):
    '''Muestra una trayectoria discreta en FrozenLake como animación HTML.'''
    env_kwargs = dict(env_kwargs or {})
    env = gym.make("FrozenLake-v1", render_mode="rgb_array", **env_kwargs)
    obs, info = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for t in range(max_steps):
        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        frames.append(env.render())
        if terminated or truncated:
            break

    env.close()

    if title:
        display(Markdown(f"**{title}**"))

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    im = ax.imshow(frames[0])
    label = ax.text(
        0.5,
        0.04,
        "",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
    )

    def update(frame_index):
        im.set_data(frames[frame_index])
        label.set_text(f"paso {frame_index}/{len(frames)-1} | recompensa {total_reward:.1f}")
        return im, label

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml(default_mode="once")))


## 3. Conceptos mínimos antes de jugar

Un problema de RL se describe con un ciclo:

- **Agente:** quien decide.
- **Entorno:** aquello con lo que interactúa.
- **Estado u observación:** lo que el agente puede percibir.
- **Acción:** lo que puede hacer.
- **Recompensa:** señal numérica que orienta el aprendizaje.
- **Política:** regla que decide qué acción tomar en cada situación.
- **Episodio:** intento completo desde un inicio hasta una condición de finalización.

La diferencia clave con aprendizaje supervisado es esta:

> No le damos al agente la respuesta correcta paso a paso; le damos consecuencias.

### Mapa del cuaderno

| Rol didáctico | Problema | Espacio que queremos contrastar |
|---|---|---|
| Ejemplo continuo | Péndulo invertido | Estado y acción continuos |
| Ejemplo discreto por discretización | MountainCar | Estado continuo discretizado y acciones discretas |
| Ejercicio continuo de control | LunarLander | Estado continuo, acciones discretas y recompensa compuesta |
| Ejercicio discreto puro | FrozenLake | Estado discreto y acciones discretas |

### CEM con más detalle

**Cross-Entropy Method (CEM)** no aprende valores $Q$ ni gradientes. Aprende una distribución de parámetros.

Partimos de una política parametrizada:

$$
a_t = \pi(s_t;\theta)
$$

La política depende de un vector de parámetros $\theta$. Al inicio no sabemos qué valores sirven, así que CEM mantiene una distribución:

$$
\theta \sim \mathcal{N}(\mu, \sigma)
$$

En cada iteración:

1. Muestra varios candidatos $\theta_1,\theta_2,\dots,\theta_N$.
2. Convierte cada candidato en una política.
3. Ejecuta episodios completos y calcula el retorno:

$$
J(\theta)=\mathbb{E}\left[\sum_{t=0}^{T} r_t\right]
$$

4. Ordena los candidatos por $J(\theta)$.
5. Conserva la élite, por ejemplo el mejor 20% o 30%.
6. Mueve la distribución hacia esa élite:

$$
\mu_{\text{nuevo}} = \mathrm{media}(\theta_{\text{élite}})
$$

$$
\sigma_{\text{nuevo}} = \mathrm{desv}(\theta_{\text{élite}})
$$

La intuición: CEM no pregunta “¿cuál acción fue correcta en este paso?”, sino “¿qué familias de parámetros producen episodios completos mejores?”.

### Recompensa original vs recompensa moldeada

En los modelos aprendidos podrás escoger entre:

- **Original:** usa exactamente la recompensa de Gymnasium.
- **Moldeada:** añade información de ingeniería para que el aprendizaje vea progreso antes de llegar al éxito final.

La recompensa moldeada suele entrenar más rápido, pero también puede sesgar el comportamiento. Por eso cada modelo aprendido se evalúa con **ambas métricas**.

### ¿Ejecutar todo desde el inicio?

La recomendación para clase es no entrenar todas las variantes de una sola vez. El cuaderno ejecuta una configuración recomendada por bloque y deja controles para explorar:

- `Dropdown`: objetivo original o moldeado.
- `Checkbox`: activar o desactivar shaping donde aplica.
- `Sliders`: pesos de recompensa o hiperparámetros.
- Barra de progreso: útil para mantener visible el avance si el entrenamiento tarda.

Con las configuraciones por defecto, cada entrenamiento está pensado para ser pequeño y razonable. Si una máquina tarda más, baja `episodios`, `population` o `iterations`.


### Diagrama conceptual de CEM

El diagrama se genera con `matplotlib`, así no dependemos de una imagen externa. Si prefieres una versión más estética, puedes reemplazar esta celda por una imagen generada aparte.


In [ ]:
def draw_cem_diagram():
    labels = [
        "Distribución\ninicial",
        "Muestrear\nparámetros",
        "Evaluar\nepisodios",
        "Seleccionar\nélite",
        "Actualizar\nmedia y sigma",
    ]

    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.axis("off")

    xs = np.linspace(0.08, 0.92, len(labels))
    y = 0.55
    for idx, (x, label) in enumerate(zip(xs, labels)):
        box = plt.Rectangle((x - 0.075, y - 0.16), 0.15, 0.32, fill=False, linewidth=1.6)
        ax.add_patch(box)
        ax.text(x, y, label, ha="center", va="center", fontsize=10)
        if idx < len(labels) - 1:
            ax.annotate(
                "",
                xy=(xs[idx + 1] - 0.085, y),
                xytext=(x + 0.085, y),
                arrowprops={"arrowstyle": "->", "linewidth": 1.5},
            )

    ax.annotate(
        "repetir",
        xy=(xs[1], 0.26),
        xytext=(xs[-1], 0.26),
        ha="center",
        arrowprops={"arrowstyle": "->", "linewidth": 1.2, "connectionstyle": "arc3,rad=-0.25"},
    )
    ax.set_title("CEM: búsqueda de parámetros guiada por recompensa", fontsize=12)
    plt.show()


draw_cem_diagram()


## Problema 1 — Ejemplo continuo - Péndulo invertido (`InvertedPendulum-v5`)

### Explicación del problema

El agente debe mantener un péndulo invertido cerca del equilibrio. Observa:

$$s_t = [x, \theta, \dot{x}, \dot{\theta}]$$

y aplica una fuerza horizontal continua:

$$a_t \in [-3, 3]$$

La recompensa del entorno instalado es simple:

$$
r_t =
\begin{cases}
1, & \text{si } |\theta| \le 0.2 \\
0, & \text{si el episodio termina}
\end{cases}
$$

Esto premia sobrevivir. Por eso una política que mantenga el péndulo estable durante más pasos obtiene mayor recompensa.

### Objetivos de aprendizaje

- Relacionar control clásico PD con una política heurística.
- Ver cómo una política aleatoria falla rápido.
- Observar que CEM puede aprender pesos parecidos a una regla de control.


### Código del problema

Creamos el entorno y mostramos sus espacios de observación y acción.


In [ ]:
pendulum_env_id = "InvertedPendulum-v5"

pendulum_env = gym.make(pendulum_env_id)
print(f"Entorno: {pendulum_env_id}")
print("Observación:", pendulum_env.observation_space)
print("Acciones:", pendulum_env.action_space)
pendulum_env.close()


### Recompensa y sintonización conceptual

El entorno original recompensa seguir vivo:

$$
r_t =
\begin{cases}
1, & |\theta_t| \le 0.2 \\
0, & \text{si cae o termina}
\end{cases}
$$

Para aprender podemos usar también una recompensa moldeada:

$$
r'_t =
r_t
- w_\theta |\theta_t|
- w_x |x_t|
- w_v(|\dot{x}_t|+|\dot{\theta}_t|)
- w_a a_t^2
- \mathbb{1}_{\text{caída}}p
$$

**Guía rápida de sintonización:**

- Sube $w_\theta$ si el agente tolera demasiada inclinación.
- Sube $w_x$ si el carrito se aleja mucho del centro.
- Sube $w_v$ si el movimiento se vuelve brusco.
- Sube $w_a$ si la política usa fuerzas excesivas.
- No exageres todos los pesos a la vez: puedes castigar tanto que el agente deje de explorar.


In [ ]:
def pendulum_shaped_reward(obs, action, next_obs, reward, terminated, truncated, info, step,
                           angle_weight=5.0,
                           position_weight=0.2,
                           velocity_weight=0.05,
                           action_weight=0.01,
                           fall_penalty=5.0):
    '''Recompensa moldeada para el péndulo invertido.'''
    x, theta, x_dot, theta_dot = np.asarray(next_obs, dtype=float)[:4]
    action_cost = float(np.sum(np.asarray(action, dtype=float) ** 2))
    shaped = float(reward)
    shaped -= angle_weight * abs(theta)
    shaped -= position_weight * abs(x)
    shaped -= velocity_weight * (abs(x_dot) + abs(theta_dot))
    shaped -= action_weight * action_cost
    if terminated:
        shaped -= fall_penalty
    return shaped


def pendulum_reward_preview(theta=0.05, x=0.0, x_dot=0.0, theta_dot=0.0,
                            action=0.0, alive_bonus=1.0,
                            angle_weight=5.0, position_weight=0.2,
                            velocity_weight=0.05, action_weight=0.01,
                            fall_penalty=5.0):
    terminated = abs(theta) > 0.2
    env_reward = 0.0 if terminated else alive_bonus
    shaped_reward = (
        env_reward
        - angle_weight * abs(theta)
        - position_weight * abs(x)
        - velocity_weight * (abs(x_dot) + abs(theta_dot))
        - action_weight * action ** 2
        - (fall_penalty if terminated else 0.0)
    )

    display(pd.DataFrame([{
        "theta": theta,
        "x": x,
        "recompensa_original": env_reward,
        "recompensa_moldeada": shaped_reward,
        "termina": terminated,
    }]))

    print("Lectura:")
    print("- La recompensa original solo pregunta si el péndulo sigue vivo.")
    print("- La moldeada distingue entre estar apenas vivo y estar realmente estable.")
    print("- Si el castigo por acción es alto, la política aprende a ser más suave.")


display(widgets.interactive(
    pendulum_reward_preview,
    theta=widgets.FloatSlider(value=0.05, min=-0.35, max=0.35, step=0.01, description="theta"),
    x=widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="x"),
    x_dot=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description="x_dot"),
    theta_dot=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description="theta_dot"),
    action=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="acción"),
    alive_bonus=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description="base"),
    angle_weight=widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5, description="w_theta"),
    position_weight=widgets.FloatSlider(value=0.2, min=0.0, max=5.0, step=0.1, description="w_x"),
    velocity_weight=widgets.FloatSlider(value=0.05, min=0.0, max=2.0, step=0.05, description="w_vel"),
    action_weight=widgets.FloatSlider(value=0.01, min=0.0, max=1.0, step=0.01, description="w_acc"),
    fall_penalty=widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5, description="caída"),
))


### Método 1: política aleatoria

No tiene parámetros. Sirve para establecer una línea base: qué ocurre si el agente actúa sin usar la observación.


In [ ]:
display(Markdown("#### Video: política aleatoria"))
show_episode_animation(
    pendulum_env_id,
    random_policy_factory(pendulum_env_id),
    seed=30,
    max_steps=180,
    fps=30,
    frame_skip=2,
    pad_to_frames=180,
    done_label="CAÍDA",
)


### Método 2: política heurística PD

La heurística combina posición, ángulo y velocidades:

$$
a =
\mathrm{clip}
\left(
s(k_p\theta + k_d\dot{\theta} + k_xx + k_v\dot{x}),
-a_{max},
a_{max}
\right)
$$

**Guía rápida de sintonización antes de tocar sliders:**

- Si se cae por no reaccionar a tiempo, sube `kp` o `kd`.
- Si el carrito se va demasiado lejos, sube `kx`.
- Si oscila mucho, sube `kd` o `kv` poco a poco.
- Si todo empeora de inmediato, cambia `sign`: la fuerza puede estar empujando al lado contrario.
- Si la acción se vuelve demasiado agresiva, baja `force_limit`.


In [ ]:
def inverted_pendulum_pd_policy(kp=8.0, kd=1.5, kx=0.5, kv=0.2, sign=1.0, force_limit=3.0):
    '''Política heurística tipo PD para mantener el péndulo cerca del equilibrio.'''
    def policy(obs):
        obs = np.asarray(obs, dtype=float)
        x = obs[0]
        theta = obs[1]
        x_dot = obs[2]
        theta_dot = obs[3]
        u = sign * (kp * theta + kd * theta_dot + kx * x + kv * x_dot)
        return np.array([np.clip(u, -force_limit, force_limit)], dtype=np.float32)
    return policy


def demo_pendulo(kp=8.0, kd=1.5, kx=0.5, kv=0.2, sign=1.0, episodios=3, max_steps=300, semilla=10):
    t0 = time.perf_counter()
    random_policy = random_policy_factory(pendulum_env_id)
    guided_policy = inverted_pendulum_pd_policy(kp=kp, kd=kd, kx=kx, kv=kv, sign=sign)

    df_random = evaluate_policy(pendulum_env_id, random_policy, episodes=episodios, max_steps=max_steps, seed=semilla)
    df_guided = evaluate_policy(pendulum_env_id, guided_policy, episodes=episodios, max_steps=max_steps, seed=semilla)

    summary = pd.DataFrame({
        "política": ["aleatoria", "heurística PD"],
        "recompensa_promedio": [df_random["recompensa"].mean(), df_guided["recompensa"].mean()],
        "pasos_promedio": [df_random["pasos"].mean(), df_guided["pasos"].mean()],
    })
    display(summary)

    fig, ax = plt.subplots()
    ax.plot(df_random["episodio"], df_random["recompensa"], marker="o", label="aleatoria")
    ax.plot(df_guided["episodio"], df_guided["recompensa"], marker="o", label="heurística PD")
    ax.set_title("Péndulo: política aleatoria vs heurística")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    ax.legend()
    plt.show()
    print(f"Tiempo de ejecución: {time.perf_counter() - t0:.2f} s")


pendulum_widget = widgets.interactive(
    demo_pendulo,
    {"manual": True, "manual_name": "Evaluar heurística"},
    kp=widgets.FloatSlider(value=8.0, min=0.0, max=30.0, step=0.5, description="kp", continuous_update=False),
    kd=widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description="kd", continuous_update=False),
    kx=widgets.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description="kx", continuous_update=False),
    kv=widgets.FloatSlider(value=0.2, min=0.0, max=5.0, step=0.1, description="kv", continuous_update=False),
    sign=widgets.Dropdown(options=[1.0, -1.0], value=1.0, description="signo"),
    episodios=widgets.IntSlider(value=3, min=1, max=10, step=1, description="episodios", continuous_update=False),
    max_steps=widgets.IntSlider(value=300, min=50, max=500, step=50, description="max_steps", continuous_update=False),
    semilla=widgets.IntSlider(value=10, min=0, max=100, step=1, description="semilla", continuous_update=False),
)
display(pendulum_widget)


In [ ]:
display(Markdown("#### Video: política heurística PD con parámetros base"))
show_episode_animation(
    pendulum_env_id,
    inverted_pendulum_pd_policy(),
    seed=31,
    max_steps=180,
    fps=30,
    frame_skip=2,
    pad_to_frames=180,
    done_label="CAÍDA",
)


### Método 3: política aprendida con CEM

CEM aprende una política lineal:

$$
a = \mathrm{clip}(w^\top s + b, -3, 3)
$$

Los parámetros aprendidos son:

$$
\theta_{CEM} = [w_x, w_\theta, w_{\dot{x}}, w_{\dot{\theta}}, b]
$$

Puedes entrenar esta política con la recompensa **original** o con la recompensa **moldeada**. Después, la tabla evalúa la misma política con ambas métricas para ver si el objetivo de entrenamiento realmente mejora el desempeño observable.


In [ ]:
def pendulum_policy_from_params(params):
    '''Crea una política lineal para `InvertedPendulum-v5` desde parámetros CEM.'''
    env = gym.make(pendulum_env_id)
    action_space = env.action_space
    env.close()

    params = np.asarray(params, dtype=float)
    weights = params[:-1]
    bias = params[-1]
    low = np.asarray(action_space.low, dtype=np.float32)
    high = np.asarray(action_space.high, dtype=np.float32)

    def policy(obs):
        obs = np.asarray(obs, dtype=float)
        raw_action = np.array([np.dot(weights[: len(obs)], obs) + bias], dtype=np.float32)
        return np.clip(raw_action, low, high)

    return policy


def learn_pendulum_policy_cem(iterations=8,
                              population=10,
                              elite_fraction=0.3,
                              episodes_per_candidate=1,
                              max_steps=300,
                              reward_objective="moldeada",
                              seed=42,
                              show_progress=True):
    env = gym.make(pendulum_env_id)
    obs_dim = int(np.prod(env.observation_space.shape))
    env.close()

    mean = np.array([0.5, 8.0, 0.2, 1.5, 0.0], dtype=float)
    sigma = np.array([1.0, 5.0, 1.0, 2.0, 0.2], dtype=float)

    if len(mean) != obs_dim + 1:
        mean = np.resize(mean, obs_dim + 1)
        sigma = np.resize(sigma, obs_dim + 1)

    reward_fn = pendulum_shaped_reward if reward_objective == "moldeada" else None
    rng = np.random.default_rng(seed)
    elite_count = max(2, int(population * elite_fraction))
    history = []
    snapshots = []
    best_params = mean.copy()
    best_score = -np.inf
    progress = make_training_progress(iterations, "CEM") if show_progress else None

    for iteration in range(iterations):
        candidates = [mean]
        candidates.extend(rng.normal(mean, sigma, size=(population - 1, len(mean))))
        scored = []

        for candidate_idx, params in enumerate(candidates):
            policy = pendulum_policy_from_params(params)
            score = score_policy_objective(
                pendulum_env_id,
                policy,
                shaped_reward_fn=reward_fn,
                episodes=episodes_per_candidate,
                max_steps=max_steps,
                seed=seed + 100 * iteration + candidate_idx,
            )
            scored.append((score, np.asarray(params, dtype=float)))
            if score > best_score:
                best_score = score
                best_params = np.asarray(params, dtype=float).copy()

        scored.sort(key=lambda item: item[0], reverse=True)
        generation_score, generation_params = scored[0]
        elites = np.array([params for _, params in scored[:elite_count]])

        snapshots.append({
            "iteración": iteration,
            "recompensa": float(generation_score),
            "params": generation_params.copy(),
        })

        mean = elites.mean(axis=0)
        sigma = np.maximum(elites.std(axis=0) * 0.8, 0.05)

        history.append({
            "iteración": iteration,
            "objetivo": reward_objective,
            "mejor_generación": float(generation_score),
            "mejor_global": float(best_score),
            "promedio_elite": float(np.mean([score for score, _ in scored[:elite_count]])),
        })

        if progress is not None:
            progress.value = iteration + 1

    learned_policy = pendulum_policy_from_params(best_params)
    return learned_policy, pd.DataFrame(history), best_params, snapshots


def demo_pendulum_cem(reward_objective="moldeada",
                      iterations=8,
                      population=10,
                      episodes_per_candidate=1,
                      max_steps=300,
                      seed=42):
    global pendulum_learned_policy, pendulum_cem_history, pendulum_learned_params, pendulum_cem_snapshots

    config = {
        "reward_objective": reward_objective,
        "iterations": iterations,
        "population": population,
        "episodes_per_candidate": episodes_per_candidate,
        "max_steps": max_steps,
        "seed": seed,
    }
    display(pd.DataFrame([config]))

    pendulum_learned_policy, pendulum_cem_history, pendulum_learned_params, pendulum_cem_snapshots = learn_pendulum_policy_cem(**config)
    display(pendulum_cem_history.tail())

    display(pd.DataFrame({
        "parámetro": ["x", "theta", "x_dot", "theta_dot", "bias"],
        "valor_aprendido": pendulum_learned_params,
    }))

    display(Markdown("#### Evaluación de la política aprendida con ambas recompensas"))
    show_objective_comparison(
        pendulum_env_id,
        [(f"aprendida con {reward_objective}", pendulum_learned_policy)],
        pendulum_shaped_reward,
        episodes=5,
        max_steps=300,
        seed=seed + 1000,
    )

    show_training_history_viewer(
        pendulum_env_id,
        pendulum_cem_snapshots,
        pendulum_policy_from_params,
        seed=32,
        max_steps=180,
        fps=30,
        frame_skip=2,
        pad_to_frames=180,
        done_label="CAÍDA",
        param_names=["x", "theta", "x_dot", "theta_dot", "bias"],
    )


display(widgets.interactive(
    demo_pendulum_cem,
    {"manual": True, "manual_name": "Entrenar CEM"},
    reward_objective=widgets.Dropdown(options=["moldeada", "original"], value="moldeada", description="objetivo"),
    iterations=widgets.IntSlider(value=8, min=3, max=20, step=1, description="iteraciones", continuous_update=False),
    population=widgets.IntSlider(value=10, min=6, max=30, step=2, description="población", continuous_update=False),
    episodes_per_candidate=widgets.IntSlider(value=1, min=1, max=5, step=1, description="episodios", continuous_update=False),
    max_steps=widgets.IntSlider(value=300, min=100, max=600, step=50, description="pasos", continuous_update=False),
    seed=widgets.IntSlider(value=42, min=0, max=100, step=1, description="semilla", continuous_update=False),
))

demo_pendulum_cem()


In [ ]:
display(Markdown("#### Video: política aprendida con CEM"))
show_episode_animation(
    pendulum_env_id,
    pendulum_learned_policy,
    seed=32,
    max_steps=180,
    fps=30,
    frame_skip=2,
    pad_to_frames=180,
    done_label="CAÍDA",
)


### Reflexión del problema 1

1. Al cambiar el selector de momento del entrenamiento, ¿en qué iteración aparece por primera vez una política claramente mejor que la aleatoria?
2. Si la política aprendida final se parece mucho a la heurística PD, ¿qué aprendió realmente CEM: una idea nueva o una forma de ajustar pesos?
3. ¿Qué diferencias pequeñas ves entre una política intermedia y la final aunque ambas mantengan el péndulo en pie?
4. ¿Por qué mirar solo el resultado final puede ocultar información importante sobre el proceso de aprendizaje?


## Problema 2 — Ejemplo discreto por discretización: `MountainCar-v0`

### Explicación del problema

Un carro está atrapado en un valle. Para llegar a la cima necesita moverse primero en sentido contrario y acumular energía.

Estado:

$$
s_t = [p, v]
$$

Acciones discretas:

$$
a_t \in \{0: izquierda,\ 1: nada,\ 2: derecha\}
$$

Recompensa original:

$$
r_t = -1
$$

hasta alcanzar la meta. Esto hace que la mejor política sea la que llegue en menos pasos.

### Objetivos de aprendizaje

- Entender recompensa diferida.
- Ver por qué exploración importa.
- Comparar Q-learning tabular con una búsqueda directa de parámetros.


### Código del problema

Inspeccionamos el entorno y sus espacios.


In [ ]:
mountain_env_id = "MountainCar-v0"

mountain_env = gym.make(mountain_env_id)
print(f"Entorno: {mountain_env_id}")
print("Observación:", mountain_env.observation_space)
print("Acciones:", mountain_env.action_space)
mountain_env.close()


### Recompensa y sintonización

La recompensa original de `MountainCar-v0` es muy escasa:

$$
r_t = -1
$$

en cada paso hasta llegar a la meta. Esto enseña una idea importante: a veces una acción útil no recibe recompensa inmediata.

Para entrenar podemos usar una versión moldeada:

$$
r'_t = -1 + w_p(p_t + 0.5) + w_v|v_t|
$$

**Guía rápida de sintonización:**

- Sube $w_v$ si quieres incentivar que el carrito tome impulso.
- Sube $w_p$ si quieres premiar avanzar hacia la derecha.
- Mantén ambos moderados: si el shaping domina, el agente puede perseguir velocidad o posición sin resolver el objetivo.
- Prueba primero `w_v` antes que `w_p`; en este problema la energía suele importar más que la posición instantánea.


In [ ]:
def mountaincar_shaped_reward(obs, action, next_obs, reward, terminated, truncated, info, step,
                              position_weight=0.2,
                              velocity_weight=3.0):
    '''Recompensa moldeada para MountainCar.'''
    position, velocity = np.asarray(next_obs, dtype=float)
    return float(reward) + position_weight * (position + 0.5) + velocity_weight * abs(velocity)


def mountaincar_reward_preview(position=-0.5, velocity=0.0, position_weight=0.2, velocity_weight=3.0):
    env_reward = -1.0
    shaped_reward = env_reward + position_weight * (position + 0.5) + velocity_weight * abs(velocity)

    display(pd.DataFrame([{
        "posición": position,
        "velocidad": velocity,
        "recompensa_original": env_reward,
        "recompensa_moldeada": shaped_reward,
    }]))

    print("Lectura:")
    print("- Mayor peso de velocidad premia tomar impulso.")
    print("- Mayor peso de posición premia avanzar hacia la derecha.")
    print("- Si exageras estos pesos, puedes sesgar la conducta.")


display(widgets.interactive(
    mountaincar_reward_preview,
    position=widgets.FloatSlider(value=-0.5, min=-1.2, max=0.6, step=0.05, description="posición"),
    velocity=widgets.FloatSlider(value=0.0, min=-0.07, max=0.07, step=0.005, description="velocidad"),
    position_weight=widgets.FloatSlider(value=0.2, min=0.0, max=5.0, step=0.1, description="w_pos"),
    velocity_weight=widgets.FloatSlider(value=3.0, min=0.0, max=20.0, step=0.5, description="w_vel"),
))


### Método 1: política aleatoria

No usa estado ni recompensa. Sirve como referencia mínima.


In [ ]:
display(Markdown("#### Video: política aleatoria"))
show_episode_animation(
    mountain_env_id,
    random_policy_factory(mountain_env_id),
    seed=40,
    max_steps=200,
    fps=30,
    frame_skip=2,
)


### Método 2: heurística de momento

Regla simple:

$$
a =
\begin{cases}
2, & v \ge 0 \\
0, & v < 0
\end{cases}
$$

Empuja en el sentido de la velocidad para acumular energía.

**Guía rápida de sintonización mental:**

- Si el carrito no llega, no necesariamente hay que empujar siempre hacia la meta.
- Empujar cuesta tiempo, pero tomar impulso en la dirección correcta puede ser mejor que avanzar poco a poco.
- Esta heurística es una buena pista para entender por qué $w_v$ ayuda en la recompensa moldeada.


In [ ]:
def mountaincar_momentum_policy():
    '''Heurística: empujar en el sentido de la velocidad para acumular energía.'''
    def policy(obs):
        position, velocity = obs
        return 2 if velocity >= 0 else 0
    return policy


display(Markdown("#### Video: política heurística de momento"))
show_episode_animation(
    mountain_env_id,
    mountaincar_momentum_policy(),
    seed=41,
    max_steps=200,
    fps=30,
    frame_skip=2,
)


### Método 3: política aprendida con Q-learning tabular

Discretizamos el estado continuo y aprendemos una tabla. La actualización es:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]
$$

Esta es una forma de la **ecuación de Bellman**, una de las ideas centrales del aprendizaje por refuerzo y de la programación dinámica. La intuición es potente: el valor de actuar bien ahora depende de la recompensa inmediata **más** el mejor futuro que podemos alcanzar desde el siguiente estado.

Fuente recomendada para profundizar: [Sutton & Barto, *Reinforcement Learning: An Introduction*](http://incompleteideas.net/book/RLbook2020.pdf).

La política aprendida es:

$$
\pi(s) = \arg\max_a Q(s,a)
$$

**Guía rápida antes de tocar sliders:**

- Sube `alpha` si quieres aprender más rápido, pero bájalo si la curva se vuelve inestable.
- Sube `gamma` si el agente debe valorar consecuencias lejanas.
- Sube `epsilon_start` si quieres más exploración inicial.
- Baja `epsilon_end` si quieres que al final actúe con menos azar.
- Activa `reward shaping` cuando el aprendizaje con $r=-1$ no muestra progreso visible.


In [ ]:
@dataclass
class Discretizer:
    '''Convierte observaciones continuas en índices discretos para Q-learning tabular.'''
    low: np.ndarray
    high: np.ndarray
    bins: Tuple[int, int]

    def transform(self, obs):
        obs = np.asarray(obs, dtype=float)
        ratios = (obs - self.low) / (self.high - self.low)
        ratios = np.clip(ratios, 0, 0.999999)
        indices = (ratios * np.array(self.bins)).astype(int)
        return tuple(indices)


def train_mountaincar_qlearning(episodes=1800,
                                bins=24,
                                alpha=0.12,
                                gamma=0.99,
                                epsilon_start=1.0,
                                epsilon_end=0.03,
                                position_weight=0.2,
                                velocity_weight=3.0,
                                reward_objective="moldeada",
                                snapshot_count=7,
                                seed=0,
                                show_progress=True):
    env = gym.make(mountain_env_id)

    low = env.observation_space.low
    high = env.observation_space.high
    discretizer = Discretizer(low=low, high=high, bins=(bins, bins))
    n_actions = env.action_space.n

    Q = np.zeros((bins, bins, n_actions), dtype=float)
    rewards = []
    snapshots = []
    snapshot_points = set(np.linspace(0, episodes - 1, snapshot_count, dtype=int))
    rng = np.random.default_rng(seed)
    progress = make_training_progress(episodes, "Q-learning") if show_progress else None
    update_every = max(1, episodes // 100)

    for ep in range(episodes):
        obs, info = env.reset(seed=seed + ep)
        state = discretizer.transform(obs)
        total_reward = 0.0
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * math.exp(-4 * ep / max(1, episodes))

        for t in range(200):
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_obs, reward, terminated, truncated, info = env.step(action)

            if reward_objective == "moldeada":
                learned_reward = mountaincar_shaped_reward(
                    obs, action, next_obs, reward, terminated, truncated, info, t,
                    position_weight=position_weight,
                    velocity_weight=velocity_weight,
                )
            else:
                learned_reward = float(reward)

            next_state = discretizer.transform(next_obs)
            best_next = np.max(Q[next_state])
            Q[state + (action,)] += alpha * (learned_reward + gamma * best_next - Q[state + (action,)])

            obs = next_obs
            state = next_state
            total_reward += float(reward)

            if terminated or truncated:
                break

        rewards.append(total_reward)

        if ep in snapshot_points:
            recent = float(np.mean(rewards[-50:]))
            snapshots.append({
                "episodio": ep + 1,
                "recompensa_reciente": recent,
                "Q": Q.copy(),
                "discretizer": discretizer,
            })

        if progress is not None and ((ep + 1) % update_every == 0 or ep == episodes - 1):
            progress.value = ep + 1

    env.close()
    return Q, discretizer, np.array(rewards), snapshots


def mountaincar_policy_from_Q(Q, discretizer):
    '''Convierte una tabla Q entrenada en una política greedy.'''
    def policy(obs):
        state = discretizer.transform(obs)
        return int(np.argmax(Q[state]))
    return policy


def demo_mountaincar(episodes=1800, bins=24, alpha=0.12, gamma=0.99,
                     epsilon_start=1.0, epsilon_end=0.03,
                     reward_objective="moldeada",
                     position_weight=0.2, velocity_weight=3.0, seed=0):
    global mountain_Q, mountain_discretizer, mountain_q_rewards, mountain_q_snapshots, mountain_q_policy

    mountain_Q, mountain_discretizer, mountain_q_rewards, mountain_q_snapshots = train_mountaincar_qlearning(
        episodes=episodes,
        bins=bins,
        alpha=alpha,
        gamma=gamma,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        reward_objective=reward_objective,
        position_weight=position_weight,
        velocity_weight=velocity_weight,
        seed=seed,
    )
    mountain_q_policy = mountaincar_policy_from_Q(mountain_Q, mountain_discretizer)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(mountain_q_rewards, alpha=0.25, label="recompensa original por episodio")
    window = max(20, episodes // 40)
    ma = moving_average(mountain_q_rewards, window=window)
    ax.plot(np.arange(len(ma)) + window - 1, ma, label="promedio móvil")
    ax.set_title(f"Entrenamiento Q-learning en MountainCar ({reward_objective})")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa original total")
    ax.legend()
    plt.show()

    display(Markdown("#### Evaluación de la política aprendida con ambas recompensas"))
    show_objective_comparison(
        mountain_env_id,
        [(f"aprendida con {reward_objective}", mountain_q_policy)],
        lambda *args: mountaincar_shaped_reward(*args, position_weight=position_weight, velocity_weight=velocity_weight),
        episodes=10,
        max_steps=200,
        seed=seed + 10000,
    )

    print("Lectura rápida:")
    print("- En MountainCar, recompensas originales más cercanas a 0 suelen ser mejores.")
    print("- Si entrenas con recompensa moldeada, revisa que también mejore la recompensa original.")
    print("- Vuelve a ejecutar el video de abajo después de entrenar con otros parámetros.")


display(widgets.interactive(
    demo_mountaincar,
    {"manual": True, "manual_name": "Entrenar Q-learning"},
    episodes=widgets.IntSlider(value=1800, min=300, max=6000, step=100, description="episodios", continuous_update=False),
    bins=widgets.IntSlider(value=24, min=8, max=40, step=2, description="bins", continuous_update=False),
    alpha=widgets.FloatSlider(value=0.12, min=0.01, max=0.8, step=0.01, description="alpha", continuous_update=False),
    gamma=widgets.FloatSlider(value=0.99, min=0.80, max=0.999, step=0.001, description="gamma", continuous_update=False),
    epsilon_start=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="eps inicial", continuous_update=False),
    epsilon_end=widgets.FloatSlider(value=0.03, min=0.0, max=0.5, step=0.01, description="eps final", continuous_update=False),
    reward_objective=widgets.Dropdown(options=["moldeada", "original"], value="moldeada", description="objetivo"),
    position_weight=widgets.FloatSlider(value=0.2, min=0.0, max=5.0, step=0.1, description="w_pos", continuous_update=False),
    velocity_weight=widgets.FloatSlider(value=3.0, min=0.0, max=20.0, step=0.5, description="w_vel", continuous_update=False),
    seed=widgets.IntSlider(value=0, min=0, max=100, step=1, description="semilla", continuous_update=False),
))

demo_mountaincar()

display(Markdown("#### Historial: distintas etapas de Q-learning"))
show_qlearning_history_viewer(
    mountain_env_id,
    mountain_q_snapshots,
    seed=48,
    max_steps=200,
    fps=30,
    frame_skip=2,
)


In [ ]:
display(Markdown("#### Video: política aprendida con Q-learning"))
show_episode_animation(
    mountain_env_id,
    mountain_q_policy,
    seed=42,
    max_steps=200,
    fps=30,
    frame_skip=2,
)


### Reflexión del problema 2

1. Al recorrer los momentos del entrenamiento, ¿cuándo aparece por primera vez la estrategia de tomar impulso?
2. ¿Por qué la mejor recompensa de una generación puede subir y bajar aunque el mejor resultado global mejore?
3. Compara la curva de Q-learning con los momentos de CEM: ¿qué muestra mejor cada representación?
4. Si una política aprendida se parece a la heurística de momento, ¿eso significa que no aprendió nada o que redescubrió una regla útil?


## Problema 3 — Ejercicio continuo de control: `LunarLander-v3`

### Explicación del problema

El agente controla un módulo lunar. Debe aterrizar con estabilidad, poco ángulo, baja velocidad y consumo razonable.

Estado:

$$
s_t = [x, y, v_x, v_y, \theta, \dot{\theta}, c_L, c_R]
$$

Acciones discretas:

$$
a_t \in \{0: nada,\ 1: motor\ lateral\ izquierdo,\ 2: motor\ principal,\ 3: motor\ lateral\ derecho\}
$$

La recompensa de Gymnasium usa una forma de shaping:

$$
\Phi(s) =
-100\sqrt{x^2+y^2}
-100\sqrt{v_x^2+v_y^2}
-100|\theta|
+10c_L+10c_R
$$

Luego:

$$
r_t = \Phi(s_t)-\Phi(s_{t-1}) - 0.30m_t - 0.03s_t
$$

con excepciones terminales: choque $=-100$ y aterrizaje exitoso $=+100$.

### Objetivos de aprendizaje

- Identificar objetivos en conflicto: ángulo, velocidad, posición y combustible.
- Ver que una recompensa compuesta puede producir trade-offs.
- Analizar cómo CEM ajusta parámetros de una heurística.


### Código del problema

Inspeccionamos el espacio de observaciones y acciones.


In [ ]:
lunar_env_id = "LunarLander-v3"

lunar_env = gym.make(lunar_env_id)
print(f"Entorno: {lunar_env_id}")
print("Observación:", lunar_env.observation_space)
print("Acciones:", lunar_env.action_space)
lunar_env.close()


### Recompensa y sintonización

La ecuación de LunarLander combina distancia, velocidad, ángulo, contacto y combustible. Una forma simplificada de verla es:

$$
\Phi(s) =
-100\sqrt{x^2+y^2}
-100\sqrt{v_x^2+v_y^2}
-100|\theta|
+10(c_L+c_R)
$$

$$
r_t \approx \Phi(s_t)-\Phi(s_{t-1})-0.30m_t-0.03s_t
$$

Además, el entorno añade bonos o castigos terminales por aterrizar o estrellarse.

Para entrenar también usaremos una versión moldeada parametrizable:

$$
r'_t =
r_t
-w_d\sqrt{x^2+y^2}
-w_v\sqrt{v_x^2+v_y^2}
-w_\theta |\theta|
+w_c(c_L+c_R)
-w_m m_t
-w_s s_t
$$

**Guía rápida de sintonización:**

- Sube $w_d$ si el módulo se aleja mucho de la plataforma.
- Sube $w_v$ si llega demasiado rápido.
- Sube $w_\theta$ si aterriza inclinado.
- Sube $w_c$ si quieres premiar contacto estable con las patas.
- Sube $w_m$ o $w_s$ si usa demasiado combustible.


In [ ]:
def lunar_shaped_reward(obs, action, next_obs, reward, terminated, truncated, info, step,
                        distance_weight=2.0,
                        speed_weight=2.0,
                        angle_weight=1.5,
                        contact_weight=1.0,
                        main_penalty=0.30,
                        side_penalty=0.03):
    '''Recompensa moldeada para LunarLander discreto.'''
    x, y, vx, vy, angle, angular_vel, left_contact, right_contact = np.asarray(next_obs, dtype=float)
    distance = math.sqrt(x * x + y * y)
    speed = math.sqrt(vx * vx + vy * vy)
    main_engine = 1.0 if int(action) == 2 else 0.0
    side_engine = 1.0 if int(action) in (1, 3) else 0.0
    shaped = float(reward)
    shaped -= distance_weight * distance
    shaped -= speed_weight * speed
    shaped -= angle_weight * abs(angle)
    shaped += contact_weight * (left_contact + right_contact)
    shaped -= main_penalty * main_engine
    shaped -= side_penalty * side_engine
    return shaped


def lunar_reward_preview(x=0.2, y=0.6, vx=0.0, vy=-0.2, angle=0.1, left_contact=0, right_contact=0,
                         main_engine=0, side_engine=0,
                         distance_weight=2.0, speed_weight=2.0, angle_weight=1.5, contact_weight=1.0,
                         main_penalty=0.30, side_penalty=0.03):
    distance = math.sqrt(x * x + y * y)
    speed = math.sqrt(vx * vx + vy * vy)
    shaped = (
        - distance_weight * distance
        - speed_weight * speed
        - angle_weight * abs(angle)
        + contact_weight * (left_contact + right_contact)
        - main_penalty * main_engine
        - side_penalty * side_engine
    )

    display(pd.DataFrame([{
        "distancia": distance,
        "rapidez": speed,
        "ángulo": angle,
        "contactos": left_contact + right_contact,
        "componente_moldeada": shaped,
    }]))

    print("Lectura:")
    print("- Distancia y velocidad suelen dominar la dificultad.")
    print("- Contacto premia aterrizar con patas, pero no debe tapar una mala trayectoria.")
    print("- Penalizar motores evita soluciones que solo queman combustible.")


display(widgets.interactive(
    lunar_reward_preview,
    x=widgets.FloatSlider(value=0.2, min=-1.5, max=1.5, step=0.05, description="x"),
    y=widgets.FloatSlider(value=0.6, min=0.0, max=1.5, step=0.05, description="y"),
    vx=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description="vx"),
    vy=widgets.FloatSlider(value=-0.2, min=-2.0, max=2.0, step=0.05, description="vy"),
    angle=widgets.FloatSlider(value=0.1, min=-1.0, max=1.0, step=0.05, description="ángulo"),
    left_contact=widgets.IntSlider(value=0, min=0, max=1, step=1, description="pata izq"),
    right_contact=widgets.IntSlider(value=0, min=0, max=1, step=1, description="pata der"),
    main_engine=widgets.IntSlider(value=0, min=0, max=1, step=1, description="motor ppal"),
    side_engine=widgets.IntSlider(value=0, min=0, max=1, step=1, description="motor lat"),
    distance_weight=widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.25, description="w_dist"),
    speed_weight=widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.25, description="w_vel"),
    angle_weight=widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.25, description="w_ang"),
    contact_weight=widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.25, description="w_contacto"),
    main_penalty=widgets.FloatSlider(value=0.30, min=0.0, max=2.0, step=0.05, description="p_ppal"),
    side_penalty=widgets.FloatSlider(value=0.03, min=0.0, max=1.0, step=0.01, description="p_lat"),
))


### Método 1: política aleatoria

No tiene parámetros. Sirve para ver el nivel de dificultad inicial.


In [ ]:
display(Markdown("#### Video: política aleatoria"))
show_episode_animation(
    lunar_env_id,
    random_policy_factory(lunar_env_id),
    seed=50,
    max_steps=400,
    fps=30,
    frame_skip=3,
)


### Método 2: política heurística

La heurística convierte posición y velocidad horizontal en un ángulo deseado:

$$
\theta^* = \mathrm{clip}(k_x x + k_{vx}v_x, -0.4, 0.4)
$$

Luego calcula dos necesidades:

$$
\text{angle\_todo}=k_\theta(\theta^*-\theta)-k_{\dot{\theta}}\dot{\theta}
$$

$$
\text{hover\_todo}=k_h(0.55|x|-y)-k_v v_y
$$

**Guía rápida antes de tocar sliders:**

- Sube `x_gain` o `vx_gain` si no corrige bien su desplazamiento horizontal.
- Sube `angle_gain` si corrige lento la inclinación.
- Sube `angle_vel_gain` si rota demasiado.
- Sube `hover_gain` si cae antes de estabilizarse.
- Sube `vel_gain` si llega con demasiada velocidad vertical.


In [ ]:
def lunar_heuristic_policy(angle_gain=0.5, angle_vel_gain=1.0, hover_gain=0.5, vel_gain=0.5, x_gain=0.5, vx_gain=1.0):
    '''Heurística simple para LunarLander discreto.'''
    def policy(obs):
        x, y, vx, vy, angle, angular_vel, left_contact, right_contact = obs

        angle_target = x_gain * x + vx_gain * vx
        angle_target = np.clip(angle_target, -0.4, 0.4)
        hover_target = 0.55 * abs(x)

        angle_todo = angle_gain * (angle_target - angle) - angle_vel_gain * angular_vel
        hover_todo = hover_gain * (hover_target - y) - vel_gain * vy

        if left_contact or right_contact:
            angle_todo = 0.0
            hover_todo = -vel_gain * vy

        if hover_todo > abs(angle_todo) and hover_todo > 0.05:
            return 2
        if angle_todo < -0.05:
            return 3
        if angle_todo > 0.05:
            return 1
        return 0
    return policy


def demo_lunar(angle_gain=0.5, angle_vel_gain=1.0, hover_gain=0.5, vel_gain=0.5, x_gain=0.5, vx_gain=1.0, episodios=5, max_steps=600, seed=20):
    random_policy = random_policy_factory(lunar_env_id)
    heuristic_policy = lunar_heuristic_policy(
        angle_gain=angle_gain,
        angle_vel_gain=angle_vel_gain,
        hover_gain=hover_gain,
        vel_gain=vel_gain,
        x_gain=x_gain,
        vx_gain=vx_gain,
    )

    df_random = evaluate_policy(lunar_env_id, random_policy, episodes=episodios, max_steps=max_steps, seed=seed)
    df_heur = evaluate_policy(lunar_env_id, heuristic_policy, episodes=episodios, max_steps=max_steps, seed=seed)

    summary = pd.DataFrame({
        "política": ["aleatoria", "heurística simple"],
        "recompensa_promedio": [df_random["recompensa"].mean(), df_heur["recompensa"].mean()],
        "pasos_promedio": [df_random["pasos"].mean(), df_heur["pasos"].mean()],
    })
    display(summary)

    fig, ax = plt.subplots()
    ax.plot(df_random["episodio"], df_random["recompensa"], marker="o", label="aleatoria")
    ax.plot(df_heur["episodio"], df_heur["recompensa"], marker="o", label="heurística")
    ax.set_title("LunarLander: política aleatoria vs heurística")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    ax.legend()
    plt.show()


display(widgets.interactive(
    demo_lunar,
    {"manual": True, "manual_name": "Evaluar LunarLander"},
    angle_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="angle_gain", continuous_update=False),
    angle_vel_gain=widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description="ang_vel", continuous_update=False),
    hover_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="hover", continuous_update=False),
    vel_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="vel", continuous_update=False),
    x_gain=widgets.FloatSlider(value=0.5, min=-1.0, max=2.0, step=0.05, description="x_gain", continuous_update=False),
    vx_gain=widgets.FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.05, description="vx_gain", continuous_update=False),
    episodios=widgets.IntSlider(value=5, min=1, max=20, step=1, description="episodios", continuous_update=False),
    max_steps=widgets.IntSlider(value=600, min=100, max=1000, step=100, description="max_steps", continuous_update=False),
    seed=widgets.IntSlider(value=20, min=0, max=100, step=1, description="semilla", continuous_update=False),
))


In [ ]:
display(Markdown("#### Video: política heurística"))
show_episode_animation(
    lunar_env_id,
    lunar_heuristic_policy(),
    seed=51,
    max_steps=400,
    fps=30,
    frame_skip=3,
)


### Método 3: política aprendida con CEM

CEM ajusta los parámetros de la heurística:

$$
\theta_{CEM} = [k_\theta, k_{\dot{\theta}}, k_h, k_v, k_x, k_{vx}]
$$

Esto conecta diseño humano con búsqueda automática: la forma de la política viene de la heurística, pero los pesos se ajustan por recompensa.


In [ ]:
def lunar_policy_from_params(params):
    '''Convierte un vector de parámetros CEM en una heurística LunarLander.'''
    return lunar_heuristic_policy(*np.asarray(params, dtype=float))


def learn_lunar_policy_cem(iterations=6,
                           population=10,
                           elite_fraction=0.3,
                           episodes_per_candidate=1,
                           max_steps=600,
                           reward_objective="moldeada",
                           seed=30,
                           show_progress=True):
    rng = np.random.default_rng(seed)
    mean = np.array([0.5, 1.0, 0.5, 0.5, 0.5, 1.0], dtype=float)
    sigma = np.array([0.5, 0.7, 0.5, 0.5, 0.7, 0.8], dtype=float)
    bounds = np.array([
        [0.0, 2.5],
        [0.0, 3.0],
        [0.0, 2.0],
        [0.0, 2.0],
        [-1.0, 2.5],
        [-1.0, 3.5],
    ])
    reward_fn = lunar_shaped_reward if reward_objective == "moldeada" else None
    elite_count = max(2, int(population * elite_fraction))
    history = []
    snapshots = []
    best_params = mean.copy()
    best_score = -np.inf
    progress = make_training_progress(iterations, "CEM") if show_progress else None

    for iteration in range(iterations):
        candidates = [mean]
        candidates.extend(np.clip(rng.normal(mean, sigma, size=(population - 1, len(mean))), bounds[:, 0], bounds[:, 1]))
        scored = []

        for candidate_idx, params in enumerate(candidates):
            policy = lunar_policy_from_params(params)
            score = score_policy_objective(
                lunar_env_id,
                policy,
                shaped_reward_fn=reward_fn,
                episodes=episodes_per_candidate,
                max_steps=max_steps,
                seed=seed + 100 * iteration + candidate_idx,
            )
            scored.append((score, np.asarray(params, dtype=float)))
            if score > best_score:
                best_score = score
                best_params = np.asarray(params, dtype=float).copy()

        scored.sort(key=lambda item: item[0], reverse=True)
        generation_score, generation_params = scored[0]
        elites = np.array([params for _, params in scored[:elite_count]])
        mean = elites.mean(axis=0)
        sigma = np.maximum(elites.std(axis=0) * 0.8, 0.05)

        snapshots.append({
            "iteración": iteration,
            "recompensa": float(generation_score),
            "params": generation_params.copy(),
        })
        history.append({
            "iteración": iteration,
            "objetivo": reward_objective,
            "mejor_generación": float(generation_score),
            "mejor_global": float(best_score),
            "promedio_elite": float(np.mean([score for score, _ in scored[:elite_count]])),
        })

        if progress is not None:
            progress.value = iteration + 1

    return lunar_policy_from_params(best_params), pd.DataFrame(history), best_params, snapshots


def demo_lunar_cem(reward_objective="moldeada",
                   iterations=6,
                   population=10,
                   episodes_per_candidate=1,
                   max_steps=600,
                   seed=30):
    global lunar_learned_policy, lunar_cem_history, lunar_learned_params, lunar_cem_snapshots

    config = {
        "reward_objective": reward_objective,
        "iterations": iterations,
        "population": population,
        "episodes_per_candidate": episodes_per_candidate,
        "max_steps": max_steps,
        "seed": seed,
    }
    display(pd.DataFrame([config]))

    lunar_learned_policy, lunar_cem_history, lunar_learned_params, lunar_cem_snapshots = learn_lunar_policy_cem(**config)
    display(lunar_cem_history.tail())
    display(pd.DataFrame({
        "parámetro": ["angle_gain", "angle_vel_gain", "hover_gain", "vel_gain", "x_gain", "vx_gain"],
        "valor_aprendido": lunar_learned_params,
    }))

    display(Markdown("#### Evaluación de la política aprendida con ambas recompensas"))
    show_objective_comparison(
        lunar_env_id,
        [(f"aprendida con {reward_objective}", lunar_learned_policy)],
        lunar_shaped_reward,
        episodes=5,
        max_steps=600,
        seed=seed + 1000,
    )

    show_training_history_viewer(
        lunar_env_id,
        lunar_cem_snapshots,
        lunar_policy_from_params,
        seed=52,
        max_steps=400,
        fps=30,
        frame_skip=3,
        param_names=["angle_gain", "angle_vel_gain", "hover_gain", "vel_gain", "x_gain", "vx_gain"],
    )


display(widgets.interactive(
    demo_lunar_cem,
    {"manual": True, "manual_name": "Entrenar CEM"},
    reward_objective=widgets.Dropdown(options=["moldeada", "original"], value="moldeada", description="objetivo"),
    iterations=widgets.IntSlider(value=6, min=3, max=15, step=1, description="iteraciones", continuous_update=False),
    population=widgets.IntSlider(value=10, min=6, max=24, step=2, description="población", continuous_update=False),
    episodes_per_candidate=widgets.IntSlider(value=1, min=1, max=4, step=1, description="episodios", continuous_update=False),
    max_steps=widgets.IntSlider(value=600, min=200, max=800, step=50, description="pasos", continuous_update=False),
    seed=widgets.IntSlider(value=30, min=0, max=100, step=1, description="semilla", continuous_update=False),
))

demo_lunar_cem()


In [ ]:
display(Markdown("#### Video: política aprendida con CEM"))
show_episode_animation(
    lunar_env_id,
    lunar_learned_policy,
    seed=52,
    max_steps=400,
    fps=30,
    frame_skip=3,
)


### Reflexión del problema 3

1. Al recorrer las iteraciones, ¿qué cambia primero: el control del ángulo, la velocidad vertical o la posición horizontal?
2. Si la política final se parece a la heurística, ¿qué parámetros aprendidos explican esa similitud?
3. ¿Hay algún momento intermedio con buena recompensa pero comportamiento visualmente riesgoso?
4. ¿Qué peligro tendría seleccionar una política solo por recompensa final sin mirar su trayectoria de aprendizaje?


## Problema 4 — Ejercicio discreto puro: `FrozenLake-v1`

En `FrozenLake`, el agente se mueve sobre una grilla. Cada estado es una casilla y cada acción es una dirección:

$$
s \in \{0,1,\dots,15\}, \qquad a \in \{\text{izquierda}, \text{abajo}, \text{derecha}, \text{arriba}\}
$$

La recompensa original es muy escasa:

$$
r_t =
\begin{cases}
1, & \text{si llega a la meta} \\
0, & \text{en otro caso}
\end{cases}
$$

Aquí el contraste es limpio: no discretizamos un estado continuo; el ambiente ya es discreto.


### Código del problema

Inspeccionamos los espacios del ambiente. Usamos el mapa `4x4` con hielo resbaloso para que la política tenga que lidiar con incertidumbre.


In [ ]:
frozen_env_id = "FrozenLake-v1"
frozen_env_kwargs = {"map_name": "4x4", "is_slippery": True}

env = gym.make(frozen_env_id, **frozen_env_kwargs)
print("Entorno:", frozen_env_id)
print("Observación:", env.observation_space)
print("Acciones:", env.action_space)
env.close()

FROZEN_ACTIONS = {
    0: "izquierda",
    1: "abajo",
    2: "derecha",
    3: "arriba",
}


### Recompensa y sintonización

La recompensa original solo aparece al llegar a la meta. Para ayudar al aprendizaje, podemos moldear la recompensa con distancia Manhattan a la meta:

$$
r'_t =
r_t
+ w_d(d(s_t)-d(s_{t+1}))
- p_{\text{paso}}
- \mathbb{1}_{\text{hueco}}p_{\text{hueco}}
$$

**Guía rápida de sintonización:**

- Sube $w_d$ si el agente necesita una pista más fuerte hacia la meta.
- Sube $p_{\text{paso}}$ si quieres trayectorias más cortas.
- Sube $p_{\text{hueco}}$ si cae demasiado en huecos.
- Si el agente se vuelve demasiado conservador, baja $p_{\text{hueco}}$ o usa menos resbalosidad.


In [ ]:
FROZEN_DESC = np.asarray([
    list("SFFF"),
    list("FHFH"),
    list("FFFH"),
    list("HFFG"),
])
FROZEN_SIZE = 4
FROZEN_GOAL = 15
FROZEN_HOLES = {5, 7, 11, 12}


def frozenlake_distance_to_goal(state, size=FROZEN_SIZE):
    row, col = divmod(int(state), size)
    goal_row, goal_col = divmod(FROZEN_GOAL, size)
    return abs(goal_row - row) + abs(goal_col - col)


def frozenlake_shaped_reward(obs, action, next_obs, reward, terminated, truncated, info, step,
                             distance_weight=0.08,
                             step_penalty=0.01,
                             hole_penalty=0.25):
    '''Recompensa moldeada para FrozenLake.'''
    progress = frozenlake_distance_to_goal(obs) - frozenlake_distance_to_goal(next_obs)
    shaped = float(reward) + distance_weight * progress - step_penalty
    if int(next_obs) in FROZEN_HOLES:
        shaped -= hole_penalty
    return shaped


def frozenlake_reward_preview(state=0, next_state=4, reward=0.0,
                              distance_weight=0.08, step_penalty=0.01, hole_penalty=0.25):
    shaped = frozenlake_shaped_reward(
        state,
        1,
        next_state,
        reward,
        int(next_state) in FROZEN_HOLES or int(next_state) == FROZEN_GOAL,
        False,
        {},
        0,
        distance_weight=distance_weight,
        step_penalty=step_penalty,
        hole_penalty=hole_penalty,
    )
    display(pd.DataFrame([{
        "estado": state,
        "siguiente_estado": next_state,
        "distancia_inicial": frozenlake_distance_to_goal(state),
        "distancia_siguiente": frozenlake_distance_to_goal(next_state),
        "recompensa_original": reward,
        "recompensa_moldeada": shaped,
    }]))


display(widgets.interactive(
    frozenlake_reward_preview,
    state=widgets.IntSlider(value=0, min=0, max=15, step=1, description="estado"),
    next_state=widgets.IntSlider(value=4, min=0, max=15, step=1, description="siguiente"),
    reward=widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=1.0, description="r"),
    distance_weight=widgets.FloatSlider(value=0.08, min=0.0, max=0.5, step=0.01, description="w_dist"),
    step_penalty=widgets.FloatSlider(value=0.01, min=0.0, max=0.1, step=0.005, description="p_paso"),
    hole_penalty=widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="p_hueco"),
))


### Método 1: política aleatoria

No usa el mapa. Sirve para ver por qué una recompensa escasa puede ser difícil.


In [ ]:
display(Markdown("#### Video: política aleatoria"))
show_frozenlake_episode_animation(
    random_policy_factory(frozen_env_id),
    seed=60,
    max_steps=30,
    fps=2,
    env_kwargs=frozen_env_kwargs,
)


### Método 2: heurística de ruta

La heurística intenta seguir una ruta segura:

$$
S \rightarrow \downarrow \downarrow \rightarrow \downarrow \rightarrow \rightarrow G
$$

**Guía rápida de sintonización mental:**

- En un lago no resbaloso, una ruta fija puede funcionar.
- En un lago resbaloso, la misma ruta puede fallar por incertidumbre.
- Esto muestra por qué aprender valores esperados puede ser mejor que memorizar una ruta.


In [ ]:
def frozenlake_route_policy():
    '''Ruta manual razonable para el mapa 4x4.'''
    route_actions = {
        0: 1,   # abajo
        4: 1,   # abajo
        8: 2,   # derecha
        9: 1,   # abajo
        13: 2,  # derecha
        14: 2,  # derecha
    }

    def policy(obs):
        return route_actions.get(int(obs), 2)

    return policy


display(Markdown("#### Video: heurística de ruta"))
show_frozenlake_episode_animation(
    frozenlake_route_policy(),
    seed=61,
    max_steps=30,
    fps=2,
    env_kwargs=frozen_env_kwargs,
)


### Método 3: política aprendida con Q-learning

Como estado y acción son discretos, aquí no necesitamos discretizador. La tabla tiene forma:

$$
Q \in \mathbb{R}^{|S|\times |A|}
$$

y usamos de nuevo Bellman:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma \max_{a'}Q(s',a') - Q(s,a)\right]
$$

Puedes entrenar con recompensa original o moldeada. En este ambiente, la moldeada ayuda a combatir la recompensa escasa.


In [ ]:
def train_frozenlake_qlearning(episodes=2500,
                                alpha=0.7,
                                gamma=0.95,
                                epsilon_start=1.0,
                                epsilon_end=0.05,
                                reward_objective="moldeada",
                                distance_weight=0.08,
                                step_penalty=0.01,
                                hole_penalty=0.25,
                                seed=0,
                                show_progress=True):
    env = gym.make(frozen_env_id, **frozen_env_kwargs)
    Q = np.zeros((env.observation_space.n, env.action_space.n), dtype=float)
    rewards = []
    rng = np.random.default_rng(seed)
    progress = make_training_progress(episodes, "Q-learning") if show_progress else None
    update_every = max(1, episodes // 100)

    for ep in range(episodes):
        obs, info = env.reset(seed=seed + ep)
        total_reward = 0.0
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * math.exp(-5 * ep / max(1, episodes))

        for t in range(100):
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[int(obs)]))

            next_obs, reward, terminated, truncated, info = env.step(action)

            if reward_objective == "moldeada":
                learned_reward = frozenlake_shaped_reward(
                    obs,
                    action,
                    next_obs,
                    reward,
                    terminated,
                    truncated,
                    info,
                    t,
                    distance_weight=distance_weight,
                    step_penalty=step_penalty,
                    hole_penalty=hole_penalty,
                )
            else:
                learned_reward = float(reward)

            best_next = np.max(Q[int(next_obs)])
            Q[int(obs), int(action)] += alpha * (learned_reward + gamma * best_next - Q[int(obs), int(action)])

            obs = next_obs
            total_reward += float(reward)
            if terminated or truncated:
                break

        rewards.append(total_reward)
        if progress is not None and ((ep + 1) % update_every == 0 or ep == episodes - 1):
            progress.value = ep + 1

    env.close()
    return Q, np.array(rewards)


def frozenlake_policy_from_Q(Q):
    '''Convierte una tabla Q de FrozenLake en política greedy.'''
    def policy(obs):
        return int(np.argmax(Q[int(obs)]))
    return policy


def demo_frozenlake_qlearning(episodes=2500,
                              alpha=0.7,
                              gamma=0.95,
                              epsilon_start=1.0,
                              epsilon_end=0.05,
                              reward_objective="moldeada",
                              distance_weight=0.08,
                              step_penalty=0.01,
                              hole_penalty=0.25,
                              seed=0):
    global frozenlake_Q, frozenlake_rewards, frozenlake_learned_policy

    frozenlake_Q, frozenlake_rewards = train_frozenlake_qlearning(
        episodes=episodes,
        alpha=alpha,
        gamma=gamma,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        reward_objective=reward_objective,
        distance_weight=distance_weight,
        step_penalty=step_penalty,
        hole_penalty=hole_penalty,
        seed=seed,
    )
    frozenlake_learned_policy = frozenlake_policy_from_Q(frozenlake_Q)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(moving_average(frozenlake_rewards, window=max(20, episodes // 50)))
    ax.set_title(f"FrozenLake Q-learning ({reward_objective})")
    ax.set_xlabel("Ventana de episodios")
    ax.set_ylabel("Recompensa original promedio")
    plt.show()

    display(Markdown("#### Evaluación de la política aprendida con ambas recompensas"))
    show_objective_comparison(
        frozen_env_id,
        [(f"aprendida con {reward_objective}", frozenlake_learned_policy)],
        lambda *args: frozenlake_shaped_reward(
            *args,
            distance_weight=distance_weight,
            step_penalty=step_penalty,
            hole_penalty=hole_penalty,
        ),
        episodes=50,
        max_steps=100,
        seed=seed + 1000,
        env_kwargs=frozen_env_kwargs,
    )


display(widgets.interactive(
    demo_frozenlake_qlearning,
    {"manual": True, "manual_name": "Entrenar Q-learning"},
    episodes=widgets.IntSlider(value=2500, min=500, max=10000, step=500, description="episodios", continuous_update=False),
    alpha=widgets.FloatSlider(value=0.7, min=0.05, max=1.0, step=0.05, description="alpha", continuous_update=False),
    gamma=widgets.FloatSlider(value=0.95, min=0.50, max=0.999, step=0.01, description="gamma", continuous_update=False),
    epsilon_start=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="eps inicial", continuous_update=False),
    epsilon_end=widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description="eps final", continuous_update=False),
    reward_objective=widgets.Dropdown(options=["moldeada", "original"], value="moldeada", description="objetivo"),
    distance_weight=widgets.FloatSlider(value=0.08, min=0.0, max=0.5, step=0.01, description="w_dist", continuous_update=False),
    step_penalty=widgets.FloatSlider(value=0.01, min=0.0, max=0.1, step=0.005, description="p_paso", continuous_update=False),
    hole_penalty=widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="p_hueco", continuous_update=False),
    seed=widgets.IntSlider(value=0, min=0, max=100, step=1, description="semilla", continuous_update=False),
))

demo_frozenlake_qlearning()


In [ ]:
display(Markdown("#### Video: política aprendida con Q-learning"))
show_frozenlake_episode_animation(
    frozenlake_learned_policy,
    seed=62,
    max_steps=50,
    fps=2,
    env_kwargs=frozen_env_kwargs,
)


### Reflexión del problema 4

1. ¿Qué cambió al pasar de un estado continuo discretizado (`MountainCar`) a un estado discreto nativo (`FrozenLake`)?
2. ¿La recompensa moldeada ayudó a encontrar la meta o produjo una política demasiado cautelosa?
3. ¿Qué efecto tiene la incertidumbre del hielo sobre una heurística de ruta fija?
4. Si entrenas con recompensa original, ¿cuánto tarda en aparecer una señal útil?


## Comparación final de políticas

Después de ejecutar los problemas, esta tabla ayuda a comparar el rendimiento promedio de las políticas disponibles con la recompensa original de cada ambiente.


In [ ]:
comparison_tables = []

if "pendulum_learned_policy" in globals():
    comparison_tables.append(compare_policy_summary(
        pendulum_env_id,
        [
            ("aleatoria", random_policy_factory(pendulum_env_id)),
            ("heurística PD", inverted_pendulum_pd_policy()),
            ("aprendida CEM", pendulum_learned_policy),
        ],
        episodes=3,
        max_steps=180,
        seed=100,
    ))

if "mountain_q_policy" in globals():
    comparison_tables.append(compare_policy_summary(
        mountain_env_id,
        [
            ("aleatoria", random_policy_factory(mountain_env_id)),
            ("heurística momento", mountaincar_momentum_policy()),
            ("aprendida Q-learning", mountain_q_policy),
        ],
        episodes=3,
        max_steps=200,
        seed=100,
    ))

if "lunar_learned_policy" in globals():
    comparison_tables.append(compare_policy_summary(
        lunar_env_id,
        [
            ("aleatoria", random_policy_factory(lunar_env_id)),
            ("heurística", lunar_heuristic_policy()),
            ("aprendida CEM", lunar_learned_policy),
        ],
        episodes=3,
        max_steps=400,
        seed=100,
    ))


if "frozenlake_learned_policy" in globals():
    comparison_tables.append(compare_policy_summary(
        frozen_env_id,
        [
            ("aleatoria", random_policy_factory(frozen_env_id)),
            ("heurística de ruta", frozenlake_route_policy()),
            ("aprendida Q-learning", frozenlake_learned_policy),
        ],
        episodes=20,
        max_steps=100,
        seed=100,
    ))

if comparison_tables:
    display(pd.concat(comparison_tables, ignore_index=True))
else:
    print("Ejecuta primero los bloques de entrenamiento para generar las políticas aprendidas.")


## Reflexiones finales

1. ¿En qué problema la política aprendida aportó más frente a la heurística?
2. ¿En qué problema la recompensa moldeada ayudó más y en cuál pudo sesgar el comportamiento?
3. ¿Qué aprendiste al mirar momentos intermedios del entrenamiento que no se veía en el resultado final?
4. ¿Qué diferencia notaste entre entrenar sobre espacios continuos, discretizados y discretos?
5. ¿Qué riesgo de reward hacking aparece en alguno de los problemas?
6. ¿Qué cambiarías antes de transferir una política de simulación a un robot real?


## Human-in-the-loop y LLMs: conexión conceptual

En RL clásico, una recompensa guía el comportamiento. En sistemas modernos, el humano puede entrar de varias formas:

- dando demostraciones;
- corrigiendo acciones;
- comparando respuestas;
- expresando preferencias;
- definiendo restricciones de seguridad.

En LLMs, esta idea aparece en métodos como RLHF: el sistema no solo aprende de texto, sino también de preferencias humanas sobre qué respuestas son más útiles, seguras o alineadas con una intención.

### Preguntas

1. ¿Qué ventaja tiene usar feedback humano?
2. ¿Qué sesgos o riesgos puede introducir?
3. ¿Por qué human-in-the-loop no vuelve automáticamente seguro a un sistema?


## Cierre: formula un problema de RL

Escoge un sistema de ingeniería o mecatrónica y formula un problema de RL en términos de:

| Elemento | Tu respuesta |
|---|---|
| Agente |  |
| Entorno |  |
| Estado u observación |  |
| Acciones |  |
| Recompensa |  |
| Episodio |  |
| Riesgo de reward hacking |  |
| Restricciones de seguridad |  |
| ¿Qué simularías antes de probar en físico? |  |

### Pregunta final

¿Qué te gustaría explorar después de esta puerta de entrada: robótica, videojuegos, control, agentes, simulación, LLMs con feedback humano u otro camino?


## Declaración breve de uso de IA

Si usaste Codex, ChatGPT, Gemini, Claude u otra herramienta durante este cuaderno, registra brevemente:

1. ¿Qué herramienta usaste?
2. ¿Para qué la usaste?
3. ¿Qué respuesta o fragmento te ayudó?
4. ¿Qué verificaste manualmente?
5. ¿Qué parte del razonamiento puedes defender con tus propias palabras?
